# MediTrust — 01 Data (Phase 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/An4ss3/MediTrust/blob/main/notebooks/01_data.ipynb)

PathMNIST+ 64×64: download, integrity checks, class distributions, IID / Dirichlet (α = 0.5) partitions.
No model is trained here. All logic lives in `src/data.py`; this notebook only calls it.

- **Runtime:** CPU is enough (no GPU needed for Phase 1).
- **Approximate time:** 5–10 min, mostly the 1.1 GB download from Zenodo.
- **Memory:** about 1.5 GB of RAM for the uint8 arrays.
- **Reference:** `docs/PROTOCOLE_EXPERIMENTAL.md`, sections 1–2.

## 1. Environment

In [ ]:
import os, sys, platform, subprocess, time
IN_COLAB = "google.colab" in sys.modules
print("Colab:", IN_COLAB, "| Python", platform.python_version(), "|", platform.platform())
try:
    import torch
    print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
except ImportError:
    print("torch not installed (not needed for this notebook)")

## 2. Source code from GitHub

In [ ]:
REPO_URL = "https://github.com/An4ss3/MediTrust.git"
if IN_COLAB:
    REPO_DIR = "/content/MediTrust"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("repo:", REPO_DIR, "| commit:", COMMIT or "unknown")

## 3. Dependencies

In [ ]:
if IN_COLAB:
    # numpy, matplotlib and torch are preinstalled on Colab; this only adds what is missing.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from src import data as D
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 4. Download PathMNIST+ 64×64

Set `USE_DRIVE = True` to keep a copy of the file on Google Drive (`MyDrive/MediTrust/data/`)
and avoid downloading it again in later sessions. Reading is always done from the VM's local disk.

In [ ]:
USE_DRIVE = False
DATA_DIR = os.path.join(REPO_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
local = os.path.join(DATA_DIR, D.FILENAME)
drive = None
if IN_COLAB and USE_DRIVE:
    from google.colab import drive as gdrive
    gdrive.mount("/content/drive")
    drive = f"/content/drive/MyDrive/MediTrust/data/{D.FILENAME}"
    if os.path.exists(drive) and not os.path.exists(local):
        subprocess.run(["cp", drive, local], check=True)
t0 = time.time()
D.download(DATA_DIR)  # no-op if the file is already present; MD5-checked otherwise
if drive and not os.path.exists(drive):
    os.makedirs(os.path.dirname(drive), exist_ok=True)
    subprocess.run(["cp", local, drive], check=True)
print(f"{local}: {os.path.getsize(local) / 1e9:.2f} GB, ready in {time.time() - t0:.0f} s")

## 5. Load and check (MD5, shapes, counts, labels)

In [ ]:
t0 = time.time()
data = D.load_pathmnist(DATA_DIR, check_md5=True)  # raises on any mismatch
print(f"MD5 {D.MD5} OK, loaded in {time.time() - t0:.0f} s\n")
print(f"{'split':6} {'images':22} {'labels':10} {'pixel range'}")
for split in D.SPLITS:
    x, y = data[split]
    print(f"{split:6} {str(x.shape):22} {str(y.shape):10} [{x.min()}, {x.max()}]")
print("\nclasses:", dict(enumerate(D.CLASS_NAMES)))

## 6. Samples from the nine classes (train)

In [ ]:
rng = np.random.default_rng(0)
x, y = data["train"]
n_per = 6
fig, axes = plt.subplots(D.N_CLASSES, n_per, figsize=(n_per * 1.2, D.N_CLASSES * 1.25))
for c in range(D.N_CLASSES):
    idx = rng.choice(np.flatnonzero(y == c), n_per, replace=False)
    for j, i in enumerate(idx):
        ax = axes[c, j]
        ax.imshow(x[i]); ax.set_xticks([]); ax.set_yticks([])
    axes[c, 0].set_ylabel(f"{c}: {D.CLASS_NAMES[c]}", rotation=0, ha="right", va="center", fontsize=8)
plt.tight_layout()
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/phase1_samples.png", dpi=100, bbox_inches="tight")
plt.show()

## 7. Class distributions

The test set (CRC-VAL-HE-7K, other patients) has a clearly different class mix from train/val.
Only train and val are used for development; the test counts are shown for information.

In [ ]:
counts = {s: D.class_counts(data[s][1]) for s in D.SPLITS}
print(f"{'class':38}" + "".join(f"{s:>9}" for s in D.SPLITS))
for c in range(D.N_CLASSES):
    print(f"{c} {D.CLASS_NAMES[c]:36}" + "".join(f"{100 * counts[s][c] / counts[s].sum():8.1f}%" for s in D.SPLITS))
fig, ax = plt.subplots(figsize=(10, 3.5))
w = 0.27
for k, s in enumerate(D.SPLITS):
    ax.bar(np.arange(D.N_CLASSES) + (k - 1) * w, counts[s] / counts[s].sum(), w, label=s)
ax.set_xticks(range(D.N_CLASSES)); ax.set_xticklabels([f"{c}" for c in range(D.N_CLASSES)])
ax.set_xlabel("class"); ax.set_ylabel("share of split"); ax.legend()
plt.tight_layout(); plt.savefig("results/figures/phase1_class_distribution.png", dpi=100); plt.show()

## 8. Federated partitions (K = 4, seed 0)

Partitions are regenerated here and compared with the versioned files in `results/partitions/`.
Identical hashes show that the partitions are reproducible across machines.

In [ ]:
train_y, val_y = data["train"][1], data["val"][1]
partitions = {}
for scheme in ("IID", "DIR05"):
    p = D.make_partition(train_y, val_y, scheme, k=4, alpha=0.5, seed=0)
    D.check_partition(p["train"], len(train_y))  # disjoint + exhaustive
    D.check_partition(p["val"], len(val_y))
    path = f"results/partitions/{scheme}_k4_s0.npz"
    h = D.partition_hash(p)
    if os.path.exists(path):
        same = D.partition_hash(D.load_partition(path)) == h
        print(f"{scheme}: identical to versioned file: {same}")
        assert same, "partition differs from the versioned file"
    else:
        D.save_partition(p, path); print(f"{scheme}: saved {path}")
    partitions[scheme] = p
    print(f"  Dirichlet draws: {p['n_draws']} | train sizes {[len(i) for i in p['train']]}"
          f" | val sizes {[len(i) for i in p['val']]} | sha256 {h[:16]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, scheme in zip(axes, partitions):
    m = D.partition_class_counts(partitions[scheme]["train"], train_y)
    im = ax.imshow(m, cmap="viridis", aspect="auto")
    for (i, j), v in np.ndenumerate(m):
        ax.text(j, i, v, ha="center", va="center", fontsize=7, color="w" if v < m.max() / 2 else "k")
    ax.set_title(f"{scheme}: train images per client and class")
    ax.set_xlabel("class"); ax.set_ylabel("client"); ax.set_yticks(range(4)); ax.set_xticks(range(D.N_CLASSES))
plt.tight_layout(); plt.savefig("results/figures/phase1_partitions.png", dpi=100); plt.show()

## 9. Integrity tests (pytest, with the real dataset)

In [ ]:
env = dict(os.environ, MEDITRUST_DATA=DATA_DIR)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True, env=env)
print(r.stdout[-2000:], r.stderr[-1000:])
assert r.returncode == 0, "tests failed"

## 10. Save a lightweight summary

Writes `results/phase1_data_summary.json`. Colab never pushes to GitHub: download this file
(and the figures) and commit them locally if they differ from the versioned ones.

In [ ]:
import json
summary = {
    "run_id": "P1-data",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "commit": COMMIT,
    "platform": {"colab": IN_COLAB, "python": platform.python_version(), "numpy": np.__version__},
    "dataset": {"file": D.FILENAME, "md5": D.MD5, "md5_verified": True,
                "n_samples": {s: int(len(data[s][1])) for s in D.SPLITS},
                "class_counts": {s: counts[s].tolist() for s in D.SPLITS}},
    "partitions": {s: {"k": 4, "seed": 0, "alpha": p["alpha"], "n_draws": p["n_draws"],
                       "sha256": D.partition_hash(p),
                       "train_sizes": [len(i) for i in p["train"]], "val_sizes": [len(i) for i in p["val"]],
                       "train_class_counts": D.partition_class_counts(p["train"], train_y).tolist()}
                   for s, p in partitions.items()},
}
with open("results/phase1_data_summary.json", "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary["dataset"]["n_samples"]), "->", "results/phase1_data_summary.json")